In [97]:
#run spark.stop before running this notebook. some of the configs won't take if an active spark sessions is grabbed or running. 
#spark.stop()

In [98]:
import psutil
import time
import threading
import datetime
import json
import xgboost as xgb
import matplotlib.pyplot as plt
import duckdb
import pandas as pd
import pandas as pd

import pyarrow as pa

import duckdb

import os

from torchgen.api.cpp import return_type
from torchvision.io import read_image
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import datasets, models
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

import numpy as np
from PIL import Image
import pyarrow
from io import BytesIO
import base64

import xgboost as xgb
import pandas as pd

import pyarrow as pa
from duckdb.typing import *

In [99]:
experiment = "1ktable"
run = 5
system = "duckdb"
udf_type = "optimized"

In [100]:
con = duckdb.connect("../duckdatabase/idnet.db")

In [101]:
def predict_tensor(base64image):

    class CNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.model = torch.nn.Sequential(
                #Input = 3 x 32 x 32, Output = 32 x 32 x 32
                #new input 3x600x375 output=
                torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 32 x 32 x 32, Output = 32 x 16 x 16
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 32 x 16 x 16, Output = 64 x 16 x 16
                torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 16 x 16, Output = 64 x 8 x 8
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                torch.nn.Flatten(),

                torch.nn.Linear(96*11*18, 128),
                torch.nn.ReLU(),
                torch.nn.BatchNorm1d(128),




                #torch.nn.Linear(128, 128),
                #torch.nn.ReLU(),
                #torch.nn.BatchNorm1d(128),



                torch.nn.Linear(128, 2),
                torch.nn.ReLU(),
                #torch.nn.BatchNorm1d(2),
                #torch.nn.Dropout(0.2),


                torch.nn.Softmax(dim=1)

            )

        def forward(self, x):



            return self.model(x)

    img_data = base64.b64decode(base64image)
    image = Image.open(BytesIO(img_data))
    tensor = ToTensor()(image).float()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = CNN().to(device)
    model.load_state_dict(torch.load('/models/mymodel', map_location=torch.device(device)))
    model.eval() # Set the model to evaluation mode

    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(tensor.unsqueeze(0).to(device))
        _, y_pred = torch.max(prediction.data, 1)

    return float(y_pred)



In [102]:
con.create_function(
    name='predict_with_torch_naive',
    function=predict_tensor,
    parameters=['STRING'],
    return_type='FLOAT'
)

In [103]:
def predict_xgboost(gender, ethnicity, driver_class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs):

    xgboostmodel = xgb.XGBClassifier(enable_categorical=True)
    xgboostmodel.load_model('/models/xgboost_model.model')

    data = [
        {
            'gender': gender,
            'ethnicity': ethnicity,
            'class': driver_class,
            'eye_color': eye_color,
            'hair_color': hair_color,
            'is_donor': is_donor,
            'is_veteran': is_veteran,
            'height_inches': height_inches,
            'weight_lbs': weight_lbs
        }
    ]
    df = pd.DataFrame(data)

    df['gender'] = df['gender'].astype('category')
    df['ethnicity'] = df['ethnicity'].astype('category')
    df['class'] = df['class'].astype('category')
    df['eye_color'] = df['eye_color'].astype('category')
    df['hair_color'] = df['hair_color'].astype('category')
    df['is_donor'] = df['is_donor'].astype('category')
    df['is_veteran'] = df['is_veteran'].astype('category')
    df['height_inches'] = df['height_inches'].astype('int64')
    df['weight_lbs'] = df['weight_lbs'].astype('int64')

    #feature_np = np.array(feature_list, dtype=object)

    y_pred = xgboostmodel.predict(df)
    #print(y_pred[0])

    return float(y_pred[0])

In [104]:
con.create_function(
    name='predict_with_xgboost_naive',
    function=predict_xgboost,
    parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER', 'VARCHAR'],
    return_type='FLOAT'
)

In [105]:
def v_predict_tensor(base64image):

    class CNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.model = torch.nn.Sequential(
                #Input = 3 x 32 x 32, Output = 32 x 32 x 32
                #new input 3x600x375 output=
                torch.nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 32 x 32 x 32, Output = 32 x 16 x 16
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 32 x 16 x 16, Output = 64 x 16 x 16
                torch.nn.Conv2d(in_channels = 6, out_channels = 12, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 16 x 16, Output = 64 x 8 x 8
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 12, out_channels = 24, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 24, out_channels = 48, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                #Input = 64 x 8 x 8, Output = 64 x 8 x 8
                torch.nn.Conv2d(in_channels = 48, out_channels = 96, kernel_size = 3, padding = 'same'),
                torch.nn.ReLU(),
                #Input = 64 x 8 x 8, Output = 64 x 4 x 4
                torch.nn.MaxPool2d(kernel_size=2),

                torch.nn.Flatten(),

                torch.nn.Linear(96*11*18, 128),
                torch.nn.ReLU(),
                torch.nn.BatchNorm1d(128),




                #torch.nn.Linear(128, 128),
                #torch.nn.ReLU(),
                #torch.nn.BatchNorm1d(128),



                torch.nn.Linear(128, 2),
                torch.nn.ReLU(),
                #torch.nn.BatchNorm1d(2),
                #torch.nn.Dropout(0.2),


                torch.nn.Softmax(dim=1)

            )

        def forward(self, x):
            return self.model(x)

    #base64image_decode = base64image.apply(lambda x: base64.b64decode(x))
    #image_pil = base64image_decode.apply(lambda x: Image.open(BytesIO(x)))
    #tensor_series = image_pil.apply(lambda x: ToTensor()(x).float())

    input_data = pd.Series(base64image)

    tensor_series = input_data.apply(lambda x: ToTensor()(Image.open(BytesIO(base64.b64decode(x)))).float())
    input_tensor = tensor_series.tolist()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = CNN().to(device)
    model.load_state_dict(torch.load('/models/mymodel', map_location=torch.device(device)))
    model.eval() # Set the model to evaluation mode

    with torch.no_grad():  # Disable gradient calculation for inference
        predictions = model(torch.stack(input_tensor).to(device))
        _, y_pred = torch.max(predictions.data, 1)
    #return y_pred
    return pa.array(y_pred.squeeze().tolist())

In [106]:
con.create_function(
    name='predict_with_torch_optimized',
    function=v_predict_tensor,
    parameters=['STRING'],
    return_type='FLOAT',
    type='arrow'
)

In [107]:
def predict_xgboost_vector(gender, ethnicity, driver_class, eye_color, hair_color, is_donor, is_veteran, height_inches,
                           weight_lbs):


    xgboostmodel = xgb.XGBClassifier(enable_categorical=True)
    xgboostmodel.load_model('/models/xgboost_model.model')


    gender_ps =  pd.Series(gender, dtype="category")
    ethnicity_ps = pd.Series(ethnicity, dtype="category")
    class_ps =  pd.Series(driver_class, dtype="category")
    eye_color_ps =  pd.Series(eye_color, dtype="category")
    hair_color_ps =  pd.Series(hair_color, dtype="category")
    is_donor_ps =  pd.Series(is_donor, dtype="category")
    is_veteran_ps =  pd.Series(is_veteran, dtype="category")
    height_inches_ps =  pd.Series(height_inches, dtype="int64")
    weight_lbs_ps =  pd.Series(weight_lbs).astype("int64")


    df = pd.DataFrame( {
        'gender': gender_ps,
        'ethnicity': ethnicity_ps,
        'class': class_ps,
        'eye_color': eye_color_ps,
        'hair_color': hair_color_ps,
        'is_donor': is_donor_ps,
        'is_veteran': is_veteran_ps,
        'height_inches': height_inches_ps,
        'weight_lbs': weight_lbs_ps
        })






    y_pred = xgboostmodel.predict(df)


    return pa.array(y_pred)

In [108]:
con.create_function(
    name='predict_with_xgboost_optimized',
    function=predict_xgboost_vector,
    parameters=['VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'INTEGER', 'VARCHAR'],
    return_type='FLOAT',
    type='arrow'
)

In [109]:
def query_1():
    con.sql('select predict_with_torch_optimized(imagedata) from idnet').show()

In [110]:
def query_2():
    con.sql('with cte as (select gender, predict_with_torch_optimized(imagedata) as my_predictions from idnet) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions').show()

In [111]:

def query_3():
    con.sql('select predict_with_xgboost_optimized(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) from idnet').show()

In [112]:
def query_4():
    con.sql('with cte as (select gender, predict_with_xgboost_optimized(gender, ethnicity, class, eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) as my_predictions from idnet) select gender, my_predictions, count(my_predictions) as count_predictions from cte group by gender, my_predictions').show()

In [113]:
def query_5():
    con.sql("""with cte as (select gender, weight_lbs, predict_with_torch_optimized(imagedata) as my_predictions from idnet) select gender, my_predictions, count(my_predictions) as count_predictions from cte where cast(weight_lbs AS INTEGER) > 150 group by gender, my_predictions;""").show()

In [114]:
log=[]



def monitor_system_resources(interval, query):
    global monitoring_active
    monitoring_active = True
    print("Monitoring system resources...")

    while monitoring_active:
        now = datetime.datetime.now()
        cpu_percent = psutil.cpu_percent(interval=None) # Non-blocking call
        virtual_memory = psutil.virtual_memory()
        memory_percent = virtual_memory.percent
        #print(f"System CPU: {cpu_percent}% | System Memory: {memory_percent}%")
        time.sleep(interval)


        pid_list = []

        for proc in psutil.process_iter(['pid', 'name', 'cpu_percent', 'memory_percent', 'memory_info']):
            try:
                # Get process information
                pid = proc.info['pid']
                name = proc.info['name']
                cpu_percent_pid = proc.cpu_percent(interval=None) # Calculate CPU usage over a short interval
                memory_percent_pid = proc.info['memory_percent']
                if cpu_percent_pid > 1 or  memory_percent_pid > 1:
                    #memory_percent_pid = proc.info['memory_percent']
                    memory_info = proc.info['memory_info']
                    rss_mb = memory_info.rss / (1024 * 1024) if memory_info else 0 # Resident Set Size in MB

                    #print(f"{pid:<8} | {name:<30} | {cpu_percent_pid:<10.2f} | {memory_percent_pid:<10.2f} | {rss_mb:<15.2f}")

                    pid_entry = { "pid": pid,
                                  "name": name,
                                  "cpu_percent": cpu_percent_pid,
                                  "memory_percent": memory_percent_pid,
                                  "rss_mb": rss_mb
                                  }

                    pid_list.append(pid_entry)


            except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
                # Handle cases where a process might disappear or access is denied
                continue

        system_entry = {
            "experiment" : experiment,
            "run" : run,
            "system" : system,
            "udf" : udf_type,
            "query" : query,
            "time" : now.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3] ,
            "cpu_percent" : cpu_percent,
            "memory_percent" : memory_percent,
            "pids": pid_list

        }

        log.append(system_entry)

    print("Monitoring stopped.")


In [115]:
log = []
query = 1
# Start the monitoring thread
monitor_thread = threading.Thread(target=monitor_system_resources, args=(1,query))
monitor_thread.start()

# Run the function to be traced
query_1()

# Stop the monitoring thread
global monitoring_active
monitoring_active = False
monitor_thread.join() # Wait for the monitoring thread to finish
print("Program finished.")

query_1_log = log[:]

"""
df = pd.DataFrame(query_1_log)
df = df.set_index('time')
plt.figure(figsize=(15, 5))
plt.plot(df['cpu_percent'])

plt.title('CPU Usage')
plt.xlabel('Time')
plt.ylabel('CPU Usage (%)')
plt.show()

plt.plot(df['memory_percent'])

plt.title('Memory Usage')
plt.xlabel('Time')
plt.ylabel('Memory Usage (%)')
plt.show()
"""

with open(f"/runlogs/{experiment}_{run}_{system}_{udf_type}_{query}.json", "w") as outfile:
    json.dump(query_1_log, outfile, indent=4)

Monitoring system resources...
┌─────────────────────────────────────────┐
│ predict_with_torch_optimized(imagedata) │
│                  float                  │
├─────────────────────────────────────────┤
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                     0.0 │
│                                      ·  │
│                                      ·  │
│                                      ·  │
│                                     1.0 │
│                                     1.0 │
│                                     1.0 │
│                                     1.0 │
│                                     1.0 │
│

In [116]:
log = []
query = 2
# Start the monitoring thread
monitor_thread = threading.Thread(target=monitor_system_resources, args=(1,query))
monitor_thread.start()

# Run the function to be traced
query_2()

# Stop the monitoring thread
global monitoring_active
monitoring_active = False
monitor_thread.join() # Wait for the monitoring thread to finish
print("Program finished.")

query_2_log = log[:]

"""
df = pd.DataFrame(query_1_log)
df = df.set_index('time')
plt.figure(figsize=(15, 5))
plt.plot(df['cpu_percent'])

plt.title('CPU Usage')
plt.xlabel('Time')
plt.ylabel('CPU Usage (%)')
plt.show()

plt.plot(df['memory_percent'])

plt.title('Memory Usage')
plt.xlabel('Time')
plt.ylabel('Memory Usage (%)')
plt.show()
"""

with open(f"/runlogs/{experiment}_{run}_{system}_{udf_type}_{query}.json", "w") as outfile:
    json.dump(query_2_log, outfile, indent=4)

Monitoring system resources...
┌─────────┬────────────────┬───────────────────┐
│ gender  │ my_predictions │ count_predictions │
│ varchar │     float      │       int64       │
├─────────┼────────────────┼───────────────────┤
│ F       │            1.0 │               104 │
│ M       │            0.0 │               357 │
│ ERROR   │            1.0 │                 3 │
│ F       │            0.0 │               442 │
│ M       │            1.0 │                92 │
└─────────┴────────────────┴───────────────────┘

Monitoring stopped.
Program finished.


In [117]:
log = []
query = 3
# Start the monitoring thread
monitor_thread = threading.Thread(target=monitor_system_resources, args=(1,query))
monitor_thread.start()

# Run the function to be traced
query_3()

# Stop the monitoring thread
global monitoring_active
monitoring_active = False
monitor_thread.join() # Wait for the monitoring thread to finish
print("Program finished.")

query_3_log = log[:]

"""
df = pd.DataFrame(query_1_log)
df = df.set_index('time')
plt.figure(figsize=(15, 5))
plt.plot(df['cpu_percent'])

plt.title('CPU Usage')
plt.xlabel('Time')
plt.ylabel('CPU Usage (%)')
plt.show()

plt.plot(df['memory_percent'])

plt.title('Memory Usage')
plt.xlabel('Time')
plt.ylabel('Memory Usage (%)')
plt.show()
"""

with open(f"/runlogs/{experiment}_{run}_{system}_{udf_type}_{query}.json", "w") as outfile:
    json.dump(query_3_log, outfile, indent=4)

Monitoring system resources...
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ predict_with_xgboost_optimized(gender, ethnicity, "class", eye_color, hair_color, is_donor, is_veteran, height_inches, weight_lbs) │
│                                                               float                                                                │
├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                                                                0.0 │
│                                                                                                                                0.0 │
│                                                                                                                                0.0 │
│                       

In [118]:
log = []
query = 4
# Start the monitoring thread
monitor_thread = threading.Thread(target=monitor_system_resources, args=(1,query))
monitor_thread.start()

# Run the function to be traced
query_4()

# Stop the monitoring thread
global monitoring_active
monitoring_active = False
monitor_thread.join() # Wait for the monitoring thread to finish
print("Program finished.")

query_4_log = log[:]

"""
df = pd.DataFrame(query_1_log)
df = df.set_index('time')
plt.figure(figsize=(15, 5))
plt.plot(df['cpu_percent'])

plt.title('CPU Usage')
plt.xlabel('Time')
plt.ylabel('CPU Usage (%)')
plt.show()

plt.plot(df['memory_percent'])

plt.title('Memory Usage')
plt.xlabel('Time')
plt.ylabel('Memory Usage (%)')
plt.show()
"""

with open(f"/runlogs/{experiment}_{run}_{system}_{udf_type}_{query}.json", "w") as outfile:
    json.dump(query_4_log, outfile, indent=4)

Monitoring system resources...
┌─────────┬────────────────┬───────────────────┐
│ gender  │ my_predictions │ count_predictions │
│ varchar │     float      │       int64       │
├─────────┼────────────────┼───────────────────┤
│ M       │            1.0 │                91 │
│ F       │            1.0 │                45 │
│ ERROR   │           NULL │                 0 │
│ F       │            0.0 │               455 │
│ M       │           NULL │                 0 │
│ M       │            0.0 │               307 │
│ F       │           NULL │                 0 │
└─────────┴────────────────┴───────────────────┘

Monitoring stopped.
Program finished.


In [119]:
log = []
query = 5
# Start the monitoring thread
monitor_thread = threading.Thread(target=monitor_system_resources, args=(1, query))
monitor_thread.start()

# Run the function to be traced
query_5()

# Stop the monitoring thread
global monitoring_active
monitoring_active = False
monitor_thread.join() # Wait for the monitoring thread to finish
print("Program finished.")

query_5_log = log[:]

"""
df = pd.DataFrame(query_4_log)
df = df.set_index('time')
plt.figure(figsize=(15, 5))
plt.plot(df['cpu_percent'])

plt.title('CPU Usage')
plt.xlabel('Time')
plt.ylabel('CPU Usage (%)')
plt.show()

plt.plot(df['memory_percent'])

plt.title('Memory Usage')
plt.xlabel('Time')
plt.ylabel('Memory Usage (%)')
plt.show()
"""

with open(f"/runlogs/{experiment}_{run}_{system}_{udf_type}_{query}.json", "w") as outfile:
    json.dump(query_5_log, outfile, indent=4)

Monitoring system resources...
┌─────────┬────────────────┬───────────────────┐
│ gender  │ my_predictions │ count_predictions │
│ varchar │     float      │       int64       │
├─────────┼────────────────┼───────────────────┤
│ F       │            1.0 │                56 │
│ F       │            0.0 │               229 │
│ M       │            1.0 │                69 │
│ M       │            0.0 │               237 │
│ ERROR   │            1.0 │                 2 │
└─────────┴────────────────┴───────────────────┘

Monitoring stopped.
Program finished.


In [120]:
con.close()